# 05 — Pro Consumidor Prices | Bronze Ingestion (incremental, Auto Loader)

Ingest every Pro Consumidor "Precios Productos de Primera Necesidad, por segmento de comercialización" workbook that arrives in the raw archive, exactly once.

| Item | Value |
|---|---|
| Archive | `/Volumes/rd_economic_intelligence/bronze/proconsumidor_raw/consumer_prices/<yyyy>/<mm>/` — one folder per arrival month, files never overwritten. Each quarterly report has the same file name, so the folder keeps the versions apart. |
| File | `precios de productos de primera necesidad.xlsx` (the CSV published alongside has inconsistent delimiters; it is archived but not ingested) |
| Layout | Title on row 1 (states the report period), headers on row 3: Id, Producto, Tipo, Precio, Mes, Año, establecimiento |
| Step 1 — discover | **Auto Loader** (`binaryFile`, `trigger(availableNow=True)`) registers new workbooks in `bronze.proconsumidor_files` — see `src/shared/ingestion.py` |
| Step 2 — capture | Each registered workbook not yet in Bronze is read and appended as one snapshot (text values, title kept) |
| Target | `rd_economic_intelligence.bronze.proconsumidor_prices_raw` |

**Status:** Development — not production-ready.

In [0]:
%pip install openpyxl==3.1.5 --quiet

In [0]:
dbutils.library.restartPython()

In [0]:
from datetime import datetime, timezone
import sys, uuid, json, importlib
import pandas as pd

nb = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
sys.path.insert(0, "/Workspace" + nb.rsplit("/", 2)[0] + "/shared")
import ingestion as ING
importlib.reload(ING)

CATALOG = "rd_economic_intelligence"
RAW_PATH = f"/Volumes/{CATALOG}/bronze/proconsumidor_raw/consumer_prices/"
CHECKPOINT = f"/Volumes/{CATALOG}/bronze/proconsumidor_raw/_checkpoints/consumer_prices_files/"
FILES = f"{CATALOG}.bronze.proconsumidor_files"
TARGET = f"{CATALOG}.bronze.proconsumidor_prices_raw"
EXPECTED = ["Id", "Producto", "Tipo", "Precio", "Mes", "Año", "establecimiento"]
BATCH_ID, INGESTION_TS = str(uuid.uuid4()), datetime.now(timezone.utc)

registered = ING.register_new_files(spark, RAW_PATH, "*.xlsx", CHECKPOINT, FILES,
    comment="Bronze: register of Pro Consumidor price workbooks found in the raw archive by Auto Loader (one row per file).")
pending = ING.pending_files(spark, FILES, TARGET)
print(f"Auto Loader registered {registered} new file(s) · pending capture: {[f['_source_path'] for f in pending]}")

In [0]:
captured = {}
for f in pending:
    raw = pd.read_excel(ING.local_path(f["_source_path"]), header=None, dtype=object)
    title = str(raw.iloc[0, 0]).strip()
    header_row = next((i for i in range(10) if [str(v).strip() for v in raw.iloc[i, :7]] == EXPECTED), None)
    if header_row is None:
        raise ValueError(f"{f['_source_path']}: structural contract failed, header {EXPECTED} not found in the first 10 rows")
    data = raw.iloc[header_row + 1:, :7]
    data.columns = EXPECTED
    data = data.dropna(how="all")
    rows = [(i + 1, *[None if pd.isna(v) else str(v) for v in r]) for i, r in zip(data.index, data.itertuples(index=False))]
    df = spark.createDataFrame(rows, "_source_row_number int, id string, producto string, tipo string, precio string, mes string, anio string, establecimiento string")
    df = ING.lineage(df.withColumn("_report_title", ING.F.lit(title)), f, BATCH_ID, INGESTION_TS)
    df.write.format("delta").mode("append").saveAsTable(TARGET)  # one atomic append per file
    captured[f["_source_path"].split("consumer_prices/")[-1]] = {"rows": len(rows), "title": title}

if spark.catalog.tableExists(TARGET):
    spark.sql(f"COMMENT ON TABLE {TARGET} IS 'Bronze: Pro Consumidor basic-goods prices by retail segment, one snapshot per archived workbook, as published (text values).'")
snapshots = spark.table(TARGET).select("_source_path").distinct().count() if spark.catalog.tableExists(TARGET) else 0
dbutils.notebook.exit(json.dumps({"status": "SUCCESS" if captured else "NO_NEW_FILES", "registered_by_auto_loader": registered,
                                  "captured": captured, "snapshots_in_bronze": snapshots}, ensure_ascii=False))